# Validar os artefatos do detector de armas

Este notebook **não treina o YOLO novamente**.

O erro `NameError: melhor_modelo is not defined` ocorreu porque o
ambiente foi desconectado e a variável em memória desapareceu.
O arquivo `best.pt` preserva o treinamento.

O notebook valida:

- `best.pt`;
- `last.pt`;
- `best.onnx`;
- todos os arquivos `.tflite` existentes;
- classes e formatos de entrada/saída.

A reexportação é opcional e vem desativada.

In [ ]:
from pathlib import Path
import json

import numpy as np
import onnx
import tensorflow as tf
import torch
import ultralytics

from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.cuda.is_available())
print("TensorFlow:", tf.__version__)

In [ ]:
def encontrar_raiz_projeto() -> Path:
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for candidato in candidatos:
        if (
            (candidato / "models").exists()
            and (candidato / "configs").exists()
        ):
            return candidato.resolve()

    raise FileNotFoundError(
        "Abra o Jupyter dentro da raiz do projeto violence-edge-ai."
    )


PASTA_RAIZ = encontrar_raiz_projeto()

PASTA_PYTORCH = PASTA_RAIZ / "models" / "weapons" / "pytorch"
PASTA_ONNX = PASTA_RAIZ / "models" / "weapons" / "onnx"
PASTA_TFLITE = PASTA_RAIZ / "models" / "weapons" / "tflite"
PASTA_METRICAS = PASTA_RAIZ / "outputs" / "metrics" / "weapons"

MELHOR_MODELO = PASTA_PYTORCH / "best.pt"
ULTIMO_MODELO = PASTA_PYTORCH / "last.pt"
MODELO_ONNX = PASTA_ONNX / "best.onnx"

PASTA_METRICAS.mkdir(parents=True, exist_ok=True)

for caminho in (
    MELHOR_MODELO,
    ULTIMO_MODELO,
    MODELO_ONNX,
):
    if caminho.exists():
        print(
            caminho.relative_to(PASTA_RAIZ),
            f"{caminho.stat().st_size / 1024**2:.2f} MB",
        )
    else:
        print("AUSENTE:", caminho)

In [ ]:
if not MELHOR_MODELO.exists():
    raise FileNotFoundError(
        "models/weapons/pytorch/best.pt não foi encontrado."
    )

melhor_modelo = YOLO(str(MELHOR_MODELO))

print("Tarefa:", melhor_modelo.task)
print("Classes:", melhor_modelo.names)
print("best.pt personalizado carregado com sucesso.")

classes_esperadas = {
    "machete",
    "knife",
    "baseball_bat",
    "rifle",
    "gun",
}

classes_modelo = set(melhor_modelo.names.values())

if classes_modelo != classes_esperadas:
    raise RuntimeError(
        "As classes do best.pt não correspondem ao dataset treinado."
    )

print("best.pt personalizado validado.")

In [ ]:
if MODELO_ONNX.exists():
    modelo_onnx = onnx.load(str(MODELO_ONNX))
    onnx.checker.check_model(modelo_onnx)
    print("ONNX válido:", MODELO_ONNX)
else:
    print("ONNX ausente. O best.pt continua suficiente.")

In [ ]:
relatorio_tflite = []

arquivos_tflite = sorted(PASTA_TFLITE.glob("*.tflite"))

if not arquivos_tflite:
    print("Nenhum TFLite encontrado.")
else:
    for caminho in arquivos_tflite:
        registro = {
            "arquivo": caminho.name,
            "tamanho_mb": caminho.stat().st_size / 1024**2,
            "valido": False,
        }

        try:
            interpretador = tf.lite.Interpreter(
                model_path=str(caminho)
            )
            interpretador.allocate_tensors()

            entradas = interpretador.get_input_details()
            saidas = interpretador.get_output_details()

            registro.update(
                {
                    "valido": True,
                    "entrada_shape": entradas[0]["shape"].tolist(),
                    "entrada_dtype": str(entradas[0]["dtype"]),
                    "entrada_quantizacao": list(
                        entradas[0]["quantization"]
                    ),
                    "saida_shape": saidas[0]["shape"].tolist(),
                    "saida_dtype": str(saidas[0]["dtype"]),
                    "saida_quantizacao": list(
                        saidas[0]["quantization"]
                    ),
                }
            )

            print("\nVÁLIDO:", caminho.name)
            print("Tamanho:", f"{registro['tamanho_mb']:.2f} MB")
            print(
                "Entrada:",
                registro["entrada_shape"],
                registro["entrada_dtype"],
                registro["entrada_quantizacao"],
            )
            print(
                "Saída:",
                registro["saida_shape"],
                registro["saida_dtype"],
                registro["saida_quantizacao"],
            )

        except Exception as erro:
            registro["erro"] = repr(erro)
            print("\nINVÁLIDO:", caminho.name)
            print(erro)

        relatorio_tflite.append(registro)

caminho_relatorio = PASTA_METRICAS / "validacao_artefatos.json"

caminho_relatorio.write_text(
    json.dumps(
        {
            "classes": melhor_modelo.names,
            "tflite": relatorio_tflite,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nRelatório:", caminho_relatorio)

## Reexportação opcional

Não execute esta célula enquanto pelo menos um TFLite estiver válido.

Para INT8, o YAML precisa apontar para um dataset disponível no
computador. Caminhos `/content/...` do Colab não funcionam localmente.

In [ ]:
EXPORTAR_NOVAMENTE = False

if EXPORTAR_NOVAMENTE:
    yaml_local = PASTA_RAIZ / "configs" / "dados_armas.yaml"

    if not yaml_local.exists():
        raise FileNotFoundError(yaml_local)

    texto_yaml = yaml_local.read_text(encoding="utf-8")

    if "/content/" in texto_yaml:
        raise RuntimeError(
            "Atualize os caminhos do YAML antes da exportação INT8."
        )

    melhor_modelo = YOLO(str(MELHOR_MODELO))

    resultado = melhor_modelo.export(
        format="tflite",
        imgsz=640,
        int8=True,
        data=str(yaml_local),
        fraction=0.25,
        nms=True,
        batch=1,
    )

    print("Exportação:", resultado)
else:
    print(
        "Reexportação desativada. "
        "O treinamento não será executado novamente."
    )